<div style="background:#12314a;color:#ffffff;padding:26px 30px;border-radius:10px;border-left:7px solid #22a6a1;">
  <div style="font-size:12px;letter-spacing:2px;color:#9ee3df;">MURSHID | TECHNICAL ASSIGNMENTS</div>
  <h1 style="color:#ffffff;margin:12px 0 8px 0;font-size:30px;">Day 04 — Authorized Tool Dispatch and Access Control</h1>
  <p style="color:#d9e8f2;font-size:15px;margin:0">Allowlisted status lookups, ownership enforcement, and negative tests</p>
</div>

**Author:** Setah Saad  
**Training program:** LLM Application Engineering · SDA-AIE-213  
**Institution:** [SDAIA Academy](https://github.com/SDAIAAcademy)  
**Notebook type:** Independent practical homework  
**Documentation language:** English

---

<div style="border-bottom:2px solid #087f83;padding-bottom:5px;margin-top:20px"><h2 style="color:#12314a;margin-bottom:0">01 | Assignment overview</h2></div>

**Purpose.** Implement a small read-only tool dispatcher backed by synthetic records, validating both the tool-call schema and record ownership.

**Learning objectives**

- Allow only a known `check_status` tool name and a strict arguments schema.
- Validate reference-number syntax and data types.
- Prevent users from reading records they do not own.
- Return the same outward code for missing and unauthorized records.
- Test authorized, rejected, and malformed calls; confirm records are unchanged.

**Technical environment**

| Component | Usage |
|:--|:--|
| Runtime | Python 3 |
| Dependencies | Standard library only (`json`) |
| Dataset | Two synthetic records in memory |
| Security controls | Tool allowlist, schema checks, trusted ownership check |
| Test suite | Seven assertions plus record-integrity check |

**Execution flow**

`Receive untrusted call → Validate shape → Allowlist tool → Validate reference → Check owner → Return read-only response → Run 7 tests`

<div style="border-left:4px solid #087f83;background:#eef7f7;padding:12px 16px;margin:12px 0"><b>Scope and limitations</b><br>The records and identities are synthetic. The caller identity `trusted_user` is supplied by the trusted application layer in this demonstration; a real system must authenticate that value independently. No actual external tool or database is connected.</div>


<div style="border-bottom:2px solid #087f83;padding-bottom:5px;margin-top:20px"><h2 style="color:#12314a;margin-bottom:0">02 | Dispatcher, authorization, and test harness</h2></div>

The `dispatch(call, trusted_user)` function accepts a **data object**, not executable code. It validates the exact top-level keys, permits only `check_status`, enforces a single `reference` argument, and checks an ASCII reference format such as `REF-1001`.

After validating the request, it looks up the record and compares `record['owner']` with the trusted user. Missing records and unauthorized records both return `{'code': 'unavailable'}`, reducing information leakage through the response category.

| Scenario | Expected result |
|:--|:--|
| Valid reference owned by trusted user | `ok` with status |
| Existing reference owned by another user | `unavailable` |
| Nonexistent reference | `unavailable` |
| Unapproved tool (`delete_record`) | `tool_not_allowed` |
| Extra argument or invalid reference type | `invalid_arguments` |
| Tool-call value is not a dictionary | `invalid_call` |

`run_tool_homework()` executes **seven** test cases and asserts the source records have not changed. It prints a JSON summary with `passed`, `total`, `records_unchanged`, and case-level results.

**Expected result when assertions pass:** `passed: 7`, `total: 7`, `records_unchanged: true`. These are the test expectations encoded in the notebook, not an independently verified deployment result.


In [1]:
# Independent additional homework solution. Standard library, no model/API.
import json

RECORDS = {'REF-1001': {'owner': 'demo-user-1', 'status': 'under_review'},
           'REF-1002': {'owner': 'demo-user-2', 'status': 'approved'}}


def dispatch(call, trusted_user):
    # A supplied tool-call object is untrusted data, never executable code.
    if not isinstance(call, dict) or set(call) != {'name', 'arguments'}:
        return {'code': 'invalid_call'}
    if call['name'] != 'check_status':
        return {'code': 'tool_not_allowed'}
    args = call['arguments']
    if not isinstance(args, dict) or set(args) != {'reference'}:
        return {'code': 'invalid_arguments'}
    ref = args['reference']
    if not isinstance(ref, str) or len(ref) != 8 or not ref.startswith('REF-') or not ref[4:].isascii() or not ref[4:].isdigit():
        return {'code': 'invalid_arguments'}
    record = RECORDS.get(ref)
    # Same outward result for absent and unauthorized records.
    if record is None or record['owner'] != trusted_user:
        return {'code': 'unavailable'}
    return {'code': 'ok', 'status': record['status']}


def run_tool_homework():
    before = json.dumps(RECORDS, sort_keys=True)
    cases = [
        ({'name': 'check_status', 'arguments': {'reference': 'REF-1001'}}, {'code': 'ok', 'status': 'under_review'}),
        ({'name': 'check_status', 'arguments': {'reference': 'REF-1002'}}, {'code': 'unavailable'}),
        ({'name': 'check_status', 'arguments': {'reference': 'REF-9999'}}, {'code': 'unavailable'}),
        ({'name': 'delete_record', 'arguments': {'reference': 'REF-1001'}}, {'code': 'tool_not_allowed'}),
        ({'name': 'check_status', 'arguments': {'reference': 'REF-1001', 'owner': 'demo-user-2'}}, {'code': 'invalid_arguments'}),
        ({'name': 'check_status', 'arguments': {'reference': 1001}}, {'code': 'invalid_arguments'}),
        ('not a tool object', {'code': 'invalid_call'}),
    ]
    rows = []
    for index, (call, expected) in enumerate(cases, 1):
        actual = dispatch(call, trusted_user='demo-user-1')
        assert actual == expected, (index, actual, expected)
        rows.append({'case': index, 'result': actual, 'passed': actual == expected})
    assert json.dumps(RECORDS, sort_keys=True) == before
    result = {'mode': 'simulated_tool_protocol', 'passed': len(rows), 'total': len(cases),
              'records_unchanged': True, 'cases': rows}
    print(json.dumps(result, indent=2))
    return result


if __name__ == '__main__':
    run_tool_homework()

{
  "mode": "simulated_tool_protocol",
  "passed": 7,
  "total": 7,
  "records_unchanged": true,
  "cases": [
    {
      "case": 1,
      "result": {
        "code": "ok",
        "status": "under_review"
      },
      "passed": true
    },
    {
      "case": 2,
      "result": {
        "code": "unavailable"
      },
      "passed": true
    },
    {
      "case": 3,
      "result": {
        "code": "unavailable"
      },
      "passed": true
    },
    {
      "case": 4,
      "result": {
        "code": "tool_not_allowed"
      },
      "passed": true
    },
    {
      "case": 5,
      "result": {
        "code": "invalid_arguments"
      },
      "passed": true
    },
    {
      "case": 6,
      "result": {
        "code": "invalid_arguments"
      },
      "passed": true
    },
    {
      "case": 7,
      "result": {
        "code": "invalid_call"
      },
      "passed": true
    }
  ]
}


<div style="border-bottom:2px solid #087f83;padding-bottom:5px;margin-top:20px"><h2 style="color:#12314a;margin-bottom:0">90 | Technical review and references</h2></div>

### Engineering takeaways
- Models a narrow, read-only tool allowlist rather than arbitrary tool execution.
- Enforces ownership checks using a separately supplied trusted identity.
- Uses the same external error code for missing and unauthorized records.
- Verifies record integrity after the test suite.

### Validation boundaries
This is a simulated protocol with in-memory records. It is not an authentication system and does not demonstrate live database integration. Passing synthetic tests does not establish production security.

### Further reading
- [Python JSON reference](https://docs.python.org/3/library/json.html)
- [OWASP authorization guidance](https://cheatsheetseries.owasp.org/cheatsheets/Authorization_Cheat_Sheet.html)
- [OWASP LLM security resources](https://genai.owasp.org/)

---
*This notebook documents an educational implementation. Any reported results should be interpreted using the actual executed outputs and the limitations stated above.*
